# Clase 161 — RL: aprendizaje por recompensa (Gymnasium)

Un **agente** interactúa con un **environment**: observa un **state**, elige una **action**,
recibe un **reward** y transita al siguiente state. El objetivo es aprender una **policy**
`π(a|s)` que maximice el return acumulado.

Requiere: `numpy`; `gymnasium` opcional (si no está instalado se muestra la API correcta,
las celdas de gym no se ejecutan). API moderna de Gymnasium (Farama), NO el viejo OpenAI Gym.

## 1. Los 5 componentes y la API de Gymnasium

In [ ]:
import numpy as np
np.random.seed(42)

try:
    import gymnasium as gym
    GYM_OK = True
    print("gymnasium", gym.__version__)
except Exception:
    GYM_OK = False
    print("gymnasium no instalado -> se muestra la API correcta (celdas de gym no se ejecutan)")

if GYM_OK:
    env = gym.make("CartPole-v1")
    print("observation_space:", env.observation_space)   # Box(4,) float32
    print("action_space:     ", env.action_space)        # Discrete(2): 0=izq, 1=der
    env.close()
else:
    print("API:  env = gym.make('CartPole-v1')")
    print("      observation_space -> Box([pos, vel, angulo, vel_ang], (4,), float32)")
    print("      action_space      -> Discrete(2)  # 0=empujar izquierda, 1=empujar derecha")

## 2. Un episodio con política aleatoria

`env.reset()` devuelve `(obs, info)`. `env.step(action)` devuelve la **5-tupla** moderna
`(obs, reward, terminated, truncated, info)`. El viejo Gym devolvía 4 valores con un `done`
que mezclaba `terminated` (falló/logró) y `truncated` (se acabó el tiempo).

In [ ]:
def run_episode(env, policy, seed=0):
    obs, info = env.reset(seed=seed)
    total, done = 0.0, False
    while not done:
        action = policy(obs)
        obs, reward, terminated, truncated, info = env.step(action)
        total += reward
        done = terminated or truncated          # NO usar solo 'done' del Gym viejo
    return total

if GYM_OK:
    env = gym.make("CartPole-v1")
    random_policy = lambda obs: env.action_space.sample()
    returns = [run_episode(env, random_policy, seed=s) for s in range(10)]
    env.close()
    print(f"random policy: return medio = {np.mean(returns):.1f} (+/- {np.std(returns):.1f})")
else:
    print("run_episode() usa la 5-tupla (obs, reward, terminated, truncated, info)")
    print("random ~ 20 steps de media en CartPole")

## 3. Política heurística: empujar contra la caída

El state de CartPole es `[pos_carro, vel_carro, angulo_polo, vel_angular]`.
Una heurística simple: si el polo cae a la izquierda (`angulo < 0`) empujar a la izquierda.

In [ ]:
def heuristic_policy(obs):
    pole_angle = obs[2]
    return 0 if pole_angle < 0 else 1     # empujar hacia donde cae el polo (mete el carro debajo)

if GYM_OK:
    env = gym.make("CartPole-v1")
    returns = [run_episode(env, heuristic_policy, seed=s) for s in range(10)]
    env.close()
    print(f"heuristica: return medio = {np.mean(returns):.1f}  (deberia superar a random)")
else:
    print("heuristica esperada >= 80 steps vs ~20 de random")

## 4. Return descontado (ejecutable en numpy)

`G_t = Σ γ^k r_{t+k}`. En CartPole cada paso da `reward = 1`, así que el return crudo es
la duración del episodio. Con descuento `γ=0.99` las recompensas lejanas pesan menos.

In [ ]:
def discounted_return(rewards, gamma=0.99):
    G = np.zeros(len(rewards), dtype=float)
    running = 0.0
    for t in reversed(range(len(rewards))):
        running = rewards[t] + gamma * running
        G[t] = running
    return G

rewards = np.ones(50)                       # episodio de 50 pasos, +1 cada uno
G = discounted_return(rewards, gamma=0.99)
print(f"return sin descuento (gamma=1): {rewards.sum():.1f}")
print(f"G_0 con gamma=0.99:            {G[0]:.3f}")
print(f"G_0 con gamma=0.90:            {discounted_return(rewards, 0.90)[0]:.3f}")

## 5. On-policy vs off-policy

| | On-policy | Off-policy |
|---|---|---|
| Entrena con | datos de la **policy actual** | datos viejos (replay buffer) |
| Ejemplos | REINFORCE, A2C, **PPO** | **DQN**, SAC |
| Sample efficiency | menor | mayor |
| Estabilidad | mayor | menor (necesita trucos) |

RL sirve para juegos, robótica, navegación y RLHF de LLMs; NO para clasificación/regresión
supervisadas típicas (ahí sobra el feedback etiquetado).

## 6. Inspeccionar varios environments

In [ ]:
envs = ["CartPole-v1", "MountainCar-v0", "LunarLander-v3"]
if GYM_OK:
    for name in envs:
        e = gym.make(name)
        print(f"{name:16s} obs={e.observation_space}  act={e.action_space}")
        e.close()
else:
    print("CartPole-v1      obs=Box(4,)   act=Discrete(2)")
    print("MountainCar-v0   obs=Box(2,)   act=Discrete(3)")
    print("LunarLander-v3   obs=Box(8,)   act=Discrete(4)")
    print("(Box = continuo; Discrete(N) = N acciones discretas)")

## Ejercicios

1. Correr 100 episodios de `random`, `heuristica` y "siempre derecha"; comparar return medio
   y desviación. Verificar que la heurística supera a random.
2. Imprimir `observation_space` y `action_space` de `CartPole-v1`, `MountainCar-v0` y
   `LunarLander-v3`; identificar cuáles usan `Discrete` y cuáles `Box`.
3. Calcular `G_t` para cada timestep de un episodio con `γ=0.99` y graficar el return acumulado.
4. Usar `render_mode='rgb_array'` para guardar frames y armar un gif del mejor episodio.

## Conclusiones

- RL = agente + environment + state + action + reward, buscando una policy que maximiza el return.
- Gymnasium (Farama) es el estándar; su `step` devuelve la 5-tupla `(obs, reward, terminated, truncated, info)`.
- `terminated` (el episodio terminó por la dinámica) es distinto de `truncated` (límite de tiempo).
- El discount `γ` regula cuánto pesan las recompensas futuras.
- Una heurística sencilla ya supera a una política aleatoria: motiva aprender la policy.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios del README. Las de **núcleo numérico** son ejecutables (con `assert` de verificación); las de frameworks/servicios no instalados aquí (TF, PyTorch, diffusers, Gymnasium, GCP…) se muestran como **código real de referencia** listo para copiar en un entorno con esas dependencias.

### Ejercicio 1 — Entorno básico: 10 episodios random en CartPole

```python
import gymnasium as gym, numpy as np
env = gym.make('CartPole-v1')
duraciones = []
for _ in range(10):
    obs, _ = env.reset()
    done = trunc = False; t = 0
    while not (done or trunc):
        obs, r, done, trunc, _ = env.step(env.action_space.sample())
        t += 1
    duraciones.append(t)
print('duracion media random:', np.mean(duraciones))  # ~20-25 pasos
env.close()
```

### Ejercicio 2 — Estructura del state y de las acciones

```python
for name in ['CartPole-v1', 'MountainCar-v0', 'LunarLander-v2']:
    e = gym.make(name)
    print(name, '| obs:', e.observation_space, '| act:', e.action_space)
# CartPole: Box(4,) / Discrete(2)  |  LunarLander: Box(8,) / Discrete(4)
```

### Ejercicio 3 — Policy heurística para CartPole

La observación es `[x, x_dot, theta, theta_dot]`; empujar en el sentido del ángulo del poste lo endereza.

```python
def heuristica(obs):
    return 0 if obs[2] < 0 else 1   # theta<0 -> empujar izquierda
obs, _ = env.reset(); done = trunc = False; t = 0
while not (done or trunc):
    obs, r, done, trunc, _ = env.step(heuristica(obs)); t += 1
print('duracion heuristica:', t)   # ~40-50, mejor que random (~22)
```

### Ejercicio 4 — Return descontado `G_t` (núcleo, ejecutable)

`G_t = Σ_k γ^k r_{t+k}` es la recompensa futura descontada, base de todo RL. Lo calculamos con NumPy y verificamos contra la definición cerrada.

In [ ]:
import numpy as np

gamma = 0.99
rewards = np.ones(50)          # CartPole da +1 por paso vivo

def returns(rewards, gamma):
    G = np.zeros_like(rewards, dtype=float)
    acc = 0.0
    for t in reversed(range(len(rewards))):
        acc = rewards[t] + gamma * acc
        G[t] = acc
    return G

G = returns(rewards, gamma)
# Verificaciones cerradas
assert np.isclose(G[-1], rewards[-1])                       # ultimo paso: sin futuro
assert np.isclose(G[0], sum(gamma ** k for k in range(50))) # serie geometrica
assert np.all(np.diff(G) <= 1e-9) or np.all(np.diff(G) >= -1)  # G decrece hacia el final
print('G[0]=%.3f  G[-1]=%.3f  (suma geometrica truncada)' % (G[0], G[-1]))
print('OK: return descontado verificado')

### Ejercicio 5 — Render a `rgb_array` para armar un gif

```python
env = gym.make('CartPole-v1', render_mode='rgb_array')
frames = []
obs, _ = env.reset(); done = trunc = False
while not (done or trunc):
    frames.append(env.render())
    obs, r, done, trunc, _ = env.step(heuristica(obs))
import imageio; imageio.mimsave('cartpole.gif', frames, fps=30)
```